# conditional mapping common support

Detailed workflow for this experiment. For catalogue inputs and dependencies, see [data](../../../docs/data.ipynb). Enable `LG_RUN_RESEARCH_WORKFLOW=1` before running the calculation cells.

In [ ]:
import os
RUN_RESEARCH_WORKFLOW = os.environ.get("LG_RUN_RESEARCH_WORKFLOW") == "1"
if not RUN_RESEARCH_WORKFLOW:
    raise RuntimeError("Optional workflow disabled. Read notebooks/92_reproduce_saved_analyses.ipynb and set LG_RUN_RESEARCH_WORKFLOW=1.")

In [ ]:
from pathlib import Path
import os
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "pyproject.toml").is_file())
os.environ.setdefault("LG_PROJECT", str(ROOT / "research"))
PROJECT = Path(os.environ["LG_PROJECT"]).expanduser().resolve()
os.chdir(PROJECT)

def design_path(logical_name):
    if logical_name == "FINAL_EXPERIMENT_DESIGN_2026-10-01.md":
        return PROJECT / "out/final_stage_audit/portability_v3/original_design.txt"
    return PROJECT / logical_name

# Conditional mapping inside common support

This notebook tests whether the relation between pair observables and the mass target differs
between FLAMINGO and TNG after restricting attention to regions where the simulations overlap.
It follows the completed domain-classification analysis and uses its fixed out-of-fold scores.

The calculation is evaluation-only. It does not fit, update or select an NPE. The primary
estimands are simulation differences in conditional target mappings at the same covariates,
not another ranking of posterior models. Results are reported for the unrestricted sample and
for two central score intervals, with pooled, same-host and different-host populations kept
separate.

The main question is whether the transfer failure can be reduced to different feature support.
If conditional mappings remain different inside common support, then systems with comparable
recorded observables still imply different mass targets in the two simulations. This would
narrow the explanation, but it would not by itself identify subgrid physics, catalogue
definitions or selection effects as the cause.


## 1. Configuration

The primary interval is `0.4 <= domain score <= 0.6`. The wider `0.3--0.7` interval and the
unrestricted sample are included as sensitivity checks. A score interval is an operational
selection under the existing balanced-prior classifier, not a physical boundary.

Linear mappings provide interpretable coefficient differences. Continuous predictors are
centred and scaled using an equal-simulation reference distribution, so the adjusted
simulation gap is evaluated at common covariate values and neither catalogue dominates the
reference. Connected host components are resampled within each simulation for uncertainty.
A shared-knot additive spline fit is used as a descriptive non-linear check.


In [ ]:
RANDOM_SEED = 260925
BOOTSTRAP_REPEATS = 500
OVERLAP_HALF_WIDTHS = [0.10, 0.20, 0.50]
PRIMARY_HALF_WIDTH = 0.10
MIN_ROWS_PER_SIMULATION = 30
SPLINE_N_KNOTS = 5
SPLINE_DEGREE = 3
SPLINE_RIDGE_ALPHA = 1.0
CURVE_GRID_SIZE = 41
CURVE_REFERENCE_PER_SIMULATION = 2000
DOMAIN_RESULT_ID = "20260925T103101Z"
OUTPUT_ROOT = ""


In [ ]:
from __future__ import annotations

import hashlib
import importlib.metadata as metadata
import json
import os
import platform
import subprocess
import sys
from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.preprocessing import SplineTransformer


def find_project_root() -> Path:
    candidates = []
    if os.environ.get("LG_PROJECT"):
        candidates.append(Path(os.environ["LG_PROJECT"]))
    candidates.extend([Path.cwd(), *Path.cwd().parents])
    for candidate in candidates:
        if (candidate / "src" / "lg_npe").is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not find the FLAMINGO project. Set LG_PROJECT to the project path."
    )


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


PROJECT = find_project_root()
sys.path.insert(0, str(PROJECT / "src"))

domain_matches = list(
    (PROJECT / "out").glob(
        f"*/domain_common_support/{DOMAIN_RESULT_ID}"
    )
)
if len(domain_matches) != 1:
    raise FileNotFoundError(
        "Expected exactly one fixed domain-common-support result for "
        f"{DOMAIN_RESULT_ID}; found {len(domain_matches)}."
    )
DOMAIN_RESULT = domain_matches[0]
DOMAIN_MANIFEST = DOMAIN_RESULT / "domain_overlap_manifest.json"

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
output_base = (
    Path(OUTPUT_ROOT).resolve()
    if OUTPUT_ROOT
    else PROJECT / "out" / "domain_shift" / "conditional_mapping"
)
OUTPUT = output_base / timestamp
OUTPUT.mkdir(parents=True, exist_ok=False)

try:
    git_revision = subprocess.run(
        ["git", "-C", str(PROJECT), "rev-parse", "HEAD"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
except (FileNotFoundError, subprocess.CalledProcessError):
    git_revision = None

domain_manifest = json.loads(DOMAIN_MANIFEST.read_text(encoding="utf-8"))
domain_scores = pd.read_csv(DOMAIN_RESULT / "domain_scores_oof.csv.gz")
posterior_join = pd.read_csv(
    DOMAIN_RESULT / "posterior_domain_join.csv.gz",
    usecols=["population", "pair_index"],
)
source_evaluation_pairs = np.sort(
    posterior_join.loc[
        posterior_join["population"] == "source_heldout", "pair_index"
    ].unique()
)
if len(source_evaluation_pairs) != 21_409:
    raise RuntimeError(
        f"Expected 21,409 held-out FLAMINGO pairs; found {len(source_evaluation_pairs):,}."
    )

versions = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit-learn": sklearn.__version__,
    "matplotlib": metadata.version("matplotlib"),
}
print(f"Project: {PROJECT}")
print(f"Fixed domain result: {DOMAIN_RESULT_ID}")
print(f"Output: {OUTPUT}")
print(json.dumps(versions, indent=2))


## 2. Populations, targets and representations

The FLAMINGO population is the same 21,409-pair held-out set used for posterior evaluation;
the TNG population contains all 1,530 corrected pairs. This prevents the larger source
catalogue from silently changing between the posterior and conditional-mapping analyses.

Both targets are included. The M200c matrix includes the historical bound-subhalo ratio and
the three stellar representations. The combined bound-subhalo-mass target excludes the
bound-subhalo ratio because that would be direct target leakage. Dynamics-only rows are kept
as controls for the adjusted simulation gap, but they have no mass-feature coefficient.


In [ ]:
from lg_npe.data import load_flamingo_pair_table, load_tng_pair_table
from lg_npe.features import build_design_matrix


ANALYSES = OrderedDict([
    ("m200c_dynamics", {
        "target": "m200c", "feature_space": "dynamics", "mass_feature": "none"
    }),
    ("m200c_bound_ratio", {
        "target": "m200c", "feature_space": "bound_subhalo_ratio",
        "mass_feature": "bound_subhalo_ratio"
    }),
    ("m200c_stellar_individual", {
        "target": "m200c", "feature_space": "stellar_individual",
        "mass_feature": "stellar_individual"
    }),
    ("m200c_stellar_ratio", {
        "target": "m200c", "feature_space": "stellar_ratio",
        "mass_feature": "stellar_ratio"
    }),
    ("m200c_stellar_sum", {
        "target": "m200c", "feature_space": "stellar_sum",
        "mass_feature": "stellar_sum"
    }),
    ("bound_sum_dynamics", {
        "target": "bound_subhalo_sum", "feature_space": "dynamics",
        "mass_feature": "none"
    }),
    ("bound_sum_stellar_individual", {
        "target": "bound_subhalo_sum", "feature_space": "stellar_individual",
        "mass_feature": "stellar_individual"
    }),
    ("bound_sum_stellar_ratio", {
        "target": "bound_subhalo_sum", "feature_space": "stellar_ratio",
        "mass_feature": "stellar_ratio"
    }),
    ("bound_sum_stellar_sum", {
        "target": "bound_subhalo_sum", "feature_space": "stellar_sum",
        "mass_feature": "stellar_sum"
    }),
])

MASS_COLUMNS = {
    "none": [],
    "bound_subhalo_ratio": ["log_mass_ratio"],
    "stellar_individual": ["log_mstar_primary", "log_mstar_secondary"],
    "stellar_ratio": ["log_stellar_ratio"],
    "stellar_sum": ["log_stellar_sum"],
}
BASE_CONTROLS = ["separation", "vr", "vt"]

source_table = load_flamingo_pair_table(PROJECT)
transfer_table = load_tng_pair_table(PROJECT)
analysis_frames = {}
population_rows = []

for analysis, specification in ANALYSES.items():
    source, source_names, source_target, source_counts = build_design_matrix(
        source_table.frame,
        target=specification["target"],
        feature_block="dynamics",
        mass_feature=specification["mass_feature"],
        validity_policy="global_complete_case",
    )
    transfer, transfer_names, transfer_target, transfer_counts = build_design_matrix(
        transfer_table.frame,
        target=specification["target"],
        feature_block="dynamics",
        mass_feature=specification["mass_feature"],
        validity_policy="global_complete_case",
    )
    if source_names != transfer_names or source_target != transfer_target:
        raise RuntimeError(f"Source and transfer definitions differ for {analysis}.")

    source = source.loc[source["pair_index"].isin(source_evaluation_pairs)].copy()
    if len(source) != len(source_evaluation_pairs):
        raise RuntimeError(f"Held-out source population is incomplete for {analysis}.")

    score_columns = [
        "pair_index", "domain_score", "outer_fold", "connected_component"
    ]
    source_scores = domain_scores.loc[
        (domain_scores["feature_space"] == specification["feature_space"])
        & (domain_scores["simulation"] == "flamingo"),
        score_columns,
    ]
    transfer_scores = domain_scores.loc[
        (domain_scores["feature_space"] == specification["feature_space"])
        & (domain_scores["simulation"] == "tng"),
        score_columns,
    ]
    source = source.merge(
        source_scores, on="pair_index", how="left", validate="one_to_one"
    )
    transfer = transfer.merge(
        transfer_scores, on="pair_index", how="left", validate="one_to_one"
    )
    if source["domain_score"].isna().any() or transfer["domain_score"].isna().any():
        raise RuntimeError(f"Domain-score join failed for {analysis}.")

    source["simulation"] = "flamingo"
    transfer["simulation"] = "tng"
    source["target_value"] = source[source_target]
    transfer["target_value"] = transfer[transfer_target]
    analysis_frames[analysis] = {"flamingo": source, "tng": transfer}

    for simulation, frame, counts in (
        ("flamingo", source, source_counts),
        ("tng", transfer, transfer_counts),
    ):
        population_rows.append({
            "analysis": analysis,
            "target": specification["target"],
            "feature_space": specification["feature_space"],
            "mass_feature": specification["mass_feature"],
            "simulation": simulation,
            "n": len(frame),
            "same_host": int(frame["same_host"].sum()),
            "different_host": int((~frame["same_host"]).sum()),
            "feature_names": json.dumps(source_names),
            "target_column": source_target,
            **{f"stage_{key}": value for key, value in counts.items()},
        })

population_registry = pd.DataFrame(population_rows)
population_registry.to_csv(OUTPUT / "conditional_population_registry.csv", index=False)
population_registry[[
    "analysis", "simulation", "n", "same_host", "different_host",
    "feature_names", "target_column",
]]


## 3. Linear conditional mappings

For each simulation, the target is regressed on separation, radial velocity, tangential
velocity and the selected mass representation. The pooled fit also includes the host-class
indicator. Fits are made separately in the two simulations, using the same centring and
scaling derived from an equal-simulation reference distribution.

The adjusted gap is the TNG-minus-FLAMINGO prediction at the common reference centre. Mass
coefficient differences are reported in target dex per one equal-reference standard
deviation of the feature. They are descriptive conditional slopes under this linear form,
not causal effects.


In [ ]:
SUBSETS = OrderedDict([
    ("pooled", lambda frame: np.ones(len(frame), dtype=bool)),
    ("same_host", lambda frame: frame["same_host"].to_numpy(bool)),
    ("different_host", lambda frame: ~frame["same_host"].to_numpy(bool)),
])


def interval_label(half_width: float) -> str:
    if half_width >= 0.5:
        return "all"
    return f"{0.5 - half_width:.2f}-{0.5 + half_width:.2f}"


def select_population(
    frame: pd.DataFrame, subset: str, half_width: float
) -> pd.DataFrame:
    subset_mask = SUBSETS[subset](frame)
    overlap_mask = (
        np.abs(frame["domain_score"].to_numpy(np.float64) - 0.5)
        <= half_width + 1e-12
    )
    return frame.loc[subset_mask & overlap_mask].copy()


def equal_reference_location_scale(
    source: pd.DataFrame,
    transfer: pd.DataFrame,
    columns: list[str],
) -> tuple[np.ndarray, np.ndarray, list[str]]:
    source_values = source[columns].to_numpy(np.float64)
    transfer_values = transfer[columns].to_numpy(np.float64)
    centre = 0.5 * (
        source_values.mean(axis=0) + transfer_values.mean(axis=0)
    )
    variance = 0.5 * (
        np.mean((source_values - centre) ** 2, axis=0)
        + np.mean((transfer_values - centre) ** 2, axis=0)
    )
    keep = np.isfinite(variance) & (variance > 1e-12)
    return centre[keep], np.sqrt(variance[keep]), [
        name for name, retained in zip(columns, keep) if retained
    ]


def make_linear_design(
    frame: pd.DataFrame,
    columns: list[str],
    centre: np.ndarray,
    scale: np.ndarray,
) -> np.ndarray:
    standardized = (frame[columns].to_numpy(np.float64) - centre) / scale
    return np.column_stack([np.ones(len(frame)), standardized])


def weighted_least_squares(
    design: np.ndarray,
    target: np.ndarray,
    weights: np.ndarray | None = None,
) -> np.ndarray:
    if weights is None:
        return np.linalg.pinv(design.T @ design) @ (design.T @ target)
    weighted_design = design * weights[:, None]
    return np.linalg.pinv(design.T @ weighted_design) @ (
        design.T @ (weights * target)
    )


def prepare_linear_mapping(
    analysis: str,
    subset: str,
    half_width: float,
) -> dict | None:
    specification = ANALYSES[analysis]
    source = select_population(
        analysis_frames[analysis]["flamingo"], subset, half_width
    )
    transfer = select_population(
        analysis_frames[analysis]["tng"], subset, half_width
    )
    if min(len(source), len(transfer)) < MIN_ROWS_PER_SIMULATION:
        return None

    candidate_columns = (
        BASE_CONTROLS
        + MASS_COLUMNS[specification["mass_feature"]]
        + (["same_host"] if subset == "pooled" else [])
    )
    centre, scale, columns = equal_reference_location_scale(
        source, transfer, candidate_columns
    )
    source_design = make_linear_design(source, columns, centre, scale)
    transfer_design = make_linear_design(transfer, columns, centre, scale)
    source_target = source["target_value"].to_numpy(np.float64)
    transfer_target = transfer["target_value"].to_numpy(np.float64)
    source_beta = weighted_least_squares(source_design, source_target)
    transfer_beta = weighted_least_squares(transfer_design, transfer_target)
    return {
        "source": source,
        "transfer": transfer,
        "columns": columns,
        "centre": centre,
        "scale": scale,
        "source_design": source_design,
        "transfer_design": transfer_design,
        "source_target": source_target,
        "transfer_target": transfer_target,
        "source_beta": source_beta,
        "transfer_beta": transfer_beta,
    }


linear_summary_rows = []
linear_coefficient_rows = []

for analysis, specification in ANALYSES.items():
    for subset in SUBSETS:
        for half_width in OVERLAP_HALF_WIDTHS:
            prepared = prepare_linear_mapping(analysis, subset, half_width)
            if prepared is None:
                linear_summary_rows.append({
                    "analysis": analysis,
                    "target": specification["target"],
                    "feature_space": specification["feature_space"],
                    "subset": subset,
                    "half_width": half_width,
                    "score_interval": interval_label(half_width),
                    "status": "insufficient_rows",
                })
                continue

            source_prediction = (
                prepared["source_design"] @ prepared["source_beta"]
            )
            transfer_prediction = (
                prepared["transfer_design"] @ prepared["transfer_beta"]
            )
            source_target = prepared["source_target"]
            transfer_target = prepared["transfer_target"]
            source = prepared["source"]
            transfer = prepared["transfer"]
            adjusted_gap = (
                prepared["transfer_beta"][0] - prepared["source_beta"][0]
            )
            linear_summary_rows.append({
                "analysis": analysis,
                "target": specification["target"],
                "feature_space": specification["feature_space"],
                "subset": subset,
                "half_width": half_width,
                "score_interval": interval_label(half_width),
                "status": "ok",
                "n_flamingo": len(source),
                "n_tng": len(transfer),
                "components_flamingo": source["connected_component"].nunique(),
                "components_tng": transfer["connected_component"].nunique(),
                "raw_target_gap_tng_minus_flamingo": (
                    transfer_target.mean() - source_target.mean()
                ),
                "adjusted_gap_tng_minus_flamingo": adjusted_gap,
                "rmse_flamingo": np.sqrt(
                    np.mean((source_prediction - source_target) ** 2)
                ),
                "rmse_tng": np.sqrt(
                    np.mean((transfer_prediction - transfer_target) ** 2)
                ),
                "r2_flamingo": r2_score(source_target, source_prediction),
                "r2_tng": r2_score(transfer_target, transfer_prediction),
                "target_std_flamingo": source_target.std(),
                "target_std_tng": transfer_target.std(),
                "condition_number_flamingo": np.linalg.cond(
                    prepared["source_design"]
                ),
                "condition_number_tng": np.linalg.cond(
                    prepared["transfer_design"]
                ),
            })
            for index, term in enumerate(["intercept", *prepared["columns"]]):
                linear_coefficient_rows.append({
                    "analysis": analysis,
                    "target": specification["target"],
                    "feature_space": specification["feature_space"],
                    "subset": subset,
                    "half_width": half_width,
                    "score_interval": interval_label(half_width),
                    "term": term,
                    "coefficient_flamingo": prepared["source_beta"][index],
                    "coefficient_tng": prepared["transfer_beta"][index],
                    "difference_tng_minus_flamingo": (
                        prepared["transfer_beta"][index]
                        - prepared["source_beta"][index]
                    ),
                    "reference_centre": (
                        np.nan if index == 0 else prepared["centre"][index - 1]
                    ),
                    "reference_scale": (
                        np.nan if index == 0 else prepared["scale"][index - 1]
                    ),
                })

linear_summary = pd.DataFrame(linear_summary_rows)
linear_coefficients = pd.DataFrame(linear_coefficient_rows)
linear_summary.to_csv(OUTPUT / "linear_mapping_summary.csv", index=False)
linear_coefficients.to_csv(OUTPUT / "linear_mapping_coefficients.csv", index=False)

linear_summary.loc[
    (linear_summary["half_width"] == PRIMARY_HALF_WIDTH)
    & (linear_summary["status"] == "ok"),
    [
        "analysis", "subset", "n_flamingo", "n_tng",
        "raw_target_gap_tng_minus_flamingo",
        "adjusted_gap_tng_minus_flamingo", "rmse_flamingo", "rmse_tng",
    ],
].round(4).reset_index(drop=True)


## 4. Connected-host-component uncertainty

Connected host components are resampled with replacement within each simulation. The
selected rows, reference centre and scale are held fixed; the resampling quantifies fitting
uncertainty under the chosen population and linear specification. It does not include NPE
retraining, cosmic variance, changes to the overlap classifier or uncertainty in catalogue
construction.

The primary output is the adjusted simulation gap. Differences in every standardized slope
are also retained, with mass-feature terms marked explicitly. The different-host rows remain
exploratory because their narrow-overlap TNG samples are small.


In [ ]:
def bootstrap_row_weights(groups: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    _, inverse = np.unique(np.asarray(groups), return_inverse=True)
    n_components = int(inverse.max()) + 1
    sampled = rng.integers(0, n_components, size=n_components)
    multiplicity = np.bincount(sampled, minlength=n_components).astype(np.float64)
    return multiplicity[inverse]


rng = np.random.default_rng(RANDOM_SEED)
bootstrap_rows = []

for analysis_number, (analysis, specification) in enumerate(ANALYSES.items(), start=1):
    print(f"[{analysis_number}/{len(ANALYSES)}] {analysis}")
    mass_terms = set(MASS_COLUMNS[specification["mass_feature"]])
    for subset in SUBSETS:
        prepared = prepare_linear_mapping(
            analysis, subset, PRIMARY_HALF_WIDTH
        )
        if prepared is None:
            continue
        source_groups = prepared["source"]["connected_component"].to_numpy()
        transfer_groups = prepared["transfer"]["connected_component"].to_numpy()
        terms = ["intercept", *prepared["columns"]]

        for repeat in range(BOOTSTRAP_REPEATS):
            source_weights = bootstrap_row_weights(source_groups, rng)
            transfer_weights = bootstrap_row_weights(transfer_groups, rng)
            source_beta = weighted_least_squares(
                prepared["source_design"],
                prepared["source_target"],
                source_weights,
            )
            transfer_beta = weighted_least_squares(
                prepared["transfer_design"],
                prepared["transfer_target"],
                transfer_weights,
            )
            difference = transfer_beta - source_beta
            for index, term in enumerate(terms):
                bootstrap_rows.append({
                    "analysis": analysis,
                    "target": specification["target"],
                    "feature_space": specification["feature_space"],
                    "subset": subset,
                    "half_width": PRIMARY_HALF_WIDTH,
                    "repeat": repeat,
                    "estimand": (
                        "adjusted_gap" if term == "intercept" else "slope_difference"
                    ),
                    "term": term,
                    "is_mass_term": term in mass_terms,
                    "value": difference[index],
                })

bootstrap_draws = pd.DataFrame(bootstrap_rows)
bootstrap_draws.to_csv(
    OUTPUT / "linear_component_bootstrap_draws.csv.gz", index=False
)

bootstrap_summary = (
    bootstrap_draws.groupby(
        [
            "analysis", "target", "feature_space", "subset", "half_width",
            "estimand", "term", "is_mass_term",
        ],
        sort=False,
    )["value"]
    .agg(
        bootstrap_median="median",
        lower_95=lambda values: np.quantile(values, 0.025),
        upper_95=lambda values: np.quantile(values, 0.975),
        bootstrap_mean="mean",
        bootstrap_std="std",
        fraction_above_zero=lambda values: np.mean(values > 0),
        repeats="size",
    )
    .reset_index()
)
point_estimates = linear_coefficients.loc[
    linear_coefficients["half_width"] == PRIMARY_HALF_WIDTH,
    [
        "analysis", "target", "feature_space", "subset", "half_width",
        "term", "difference_tng_minus_flamingo",
    ],
].copy()
point_estimates["estimand"] = np.where(
    point_estimates["term"] == "intercept",
    "adjusted_gap",
    "slope_difference",
)
point_estimates = point_estimates.rename(
    columns={"difference_tng_minus_flamingo": "estimate"}
)
bootstrap_summary = bootstrap_summary.merge(
    point_estimates,
    on=[
        "analysis", "target", "feature_space", "subset", "half_width",
        "estimand", "term",
    ],
    how="left",
    validate="one_to_one",
)
if bootstrap_summary["estimate"].isna().any():
    raise RuntimeError("A bootstrap row is missing its original point estimate.")
bootstrap_summary.to_csv(
    OUTPUT / "linear_component_bootstrap_summary.csv", index=False
)

bootstrap_summary.loc[
    (bootstrap_summary["estimand"] == "adjusted_gap")
    | bootstrap_summary["is_mass_term"],
    [
        "analysis", "subset", "estimand", "term", "estimate",
        "lower_95", "upper_95", "fraction_above_zero", "repeats",
    ],
].round(4).reset_index(drop=True)


## 5. Additive spline sensitivity

The linear interaction is deliberately simple. As a sensitivity check, each continuous
predictor is expanded through a cubic spline with knots fixed from an equal-simulation
weighted distribution. The same basis is used in FLAMINGO and TNG, and separate ridge fits
are compared on the same covariate rows.

This model permits curved one-dimensional relationships while remaining additive. It does
not represent arbitrary high-order interactions. Its adjusted gaps and conditional mass
curves are descriptive checks on the linear conclusion and are not given bootstrap intervals.


In [ ]:
def weighted_quantile(
    values: np.ndarray,
    probabilities: np.ndarray,
    weights: np.ndarray,
) -> np.ndarray:
    order = np.argsort(values)
    ordered_values = np.asarray(values)[order]
    ordered_weights = np.asarray(weights, dtype=np.float64)[order]
    cumulative = np.cumsum(ordered_weights)
    cumulative /= cumulative[-1]
    return np.interp(probabilities, cumulative, ordered_values)


def equal_reference_weights(n_source: int, n_transfer: int) -> np.ndarray:
    return np.concatenate([
        np.full(n_source, 0.5 / n_source),
        np.full(n_transfer, 0.5 / n_transfer),
    ])


def prepare_spline_mapping(
    analysis: str,
    subset: str,
    half_width: float,
) -> dict | None:
    specification = ANALYSES[analysis]
    source = select_population(
        analysis_frames[analysis]["flamingo"], subset, half_width
    )
    transfer = select_population(
        analysis_frames[analysis]["tng"], subset, half_width
    )
    if min(len(source), len(transfer)) < MIN_ROWS_PER_SIMULATION:
        return None

    continuous_columns = (
        BASE_CONTROLS + MASS_COLUMNS[specification["mass_feature"]]
    )
    combined = pd.concat([source, transfer], ignore_index=True)
    reference_weights = equal_reference_weights(len(source), len(transfer))
    knot_probabilities = np.linspace(0.0, 1.0, SPLINE_N_KNOTS)
    knots = np.column_stack([
        weighted_quantile(
            combined[column].to_numpy(np.float64),
            knot_probabilities,
            reference_weights,
        )
        for column in continuous_columns
    ])
    transformer = SplineTransformer(
        n_knots=SPLINE_N_KNOTS,
        degree=SPLINE_DEGREE,
        knots=knots,
        include_bias=False,
        extrapolation="linear",
    )
    transformer.fit(combined[continuous_columns].to_numpy(np.float64))

    def transform(frame: pd.DataFrame) -> np.ndarray:
        basis = transformer.transform(
            frame[continuous_columns].to_numpy(np.float64)
        )
        if subset == "pooled":
            basis = np.column_stack([
                basis, frame["same_host"].to_numpy(np.float64)
            ])
        return basis

    source_design = transform(source)
    transfer_design = transform(transfer)
    source_model = Ridge(alpha=SPLINE_RIDGE_ALPHA, fit_intercept=True)
    transfer_model = Ridge(alpha=SPLINE_RIDGE_ALPHA, fit_intercept=True)
    source_model.fit(source_design, source["target_value"])
    transfer_model.fit(transfer_design, transfer["target_value"])
    return {
        "source": source,
        "transfer": transfer,
        "continuous_columns": continuous_columns,
        "reference_weights": reference_weights,
        "transformer": transformer,
        "transform": transform,
        "source_design": source_design,
        "transfer_design": transfer_design,
        "source_model": source_model,
        "transfer_model": transfer_model,
    }


flexible_summary_rows = []
curve_rows = []
curve_rng = np.random.default_rng(RANDOM_SEED + 1000)

for analysis, specification in ANALYSES.items():
    for subset in SUBSETS:
        for half_width in OVERLAP_HALF_WIDTHS:
            prepared = prepare_spline_mapping(analysis, subset, half_width)
            if prepared is None:
                flexible_summary_rows.append({
                    "analysis": analysis,
                    "target": specification["target"],
                    "feature_space": specification["feature_space"],
                    "subset": subset,
                    "half_width": half_width,
                    "score_interval": interval_label(half_width),
                    "status": "insufficient_rows",
                })
                continue

            source = prepared["source"]
            transfer = prepared["transfer"]
            source_model = prepared["source_model"]
            transfer_model = prepared["transfer_model"]
            source_on_source = source_model.predict(prepared["source_design"])
            transfer_on_source = transfer_model.predict(prepared["source_design"])
            source_on_transfer = source_model.predict(prepared["transfer_design"])
            transfer_on_transfer = transfer_model.predict(prepared["transfer_design"])
            adjusted_gap = 0.5 * (
                np.mean(transfer_on_source - source_on_source)
                + np.mean(transfer_on_transfer - source_on_transfer)
            )
            flexible_summary_rows.append({
                "analysis": analysis,
                "target": specification["target"],
                "feature_space": specification["feature_space"],
                "subset": subset,
                "half_width": half_width,
                "score_interval": interval_label(half_width),
                "status": "ok",
                "n_flamingo": len(source),
                "n_tng": len(transfer),
                "adjusted_gap_tng_minus_flamingo": adjusted_gap,
                "rmse_flamingo": np.sqrt(np.mean(
                    (source_on_source - source["target_value"].to_numpy()) ** 2
                )),
                "rmse_tng": np.sqrt(np.mean(
                    (transfer_on_transfer - transfer["target_value"].to_numpy()) ** 2
                )),
            })

            mass_columns = MASS_COLUMNS[specification["mass_feature"]]
            if half_width != PRIMARY_HALF_WIDTH or not mass_columns:
                continue

            source_reference = source.sample(
                n=min(CURVE_REFERENCE_PER_SIMULATION, len(source)),
                replace=False,
                random_state=RANDOM_SEED,
            )
            transfer_reference = transfer.sample(
                n=min(CURVE_REFERENCE_PER_SIMULATION, len(transfer)),
                replace=False,
                random_state=RANDOM_SEED + 1,
            )
            reference = pd.concat(
                [source_reference, transfer_reference], ignore_index=True
            )
            reference_weights = equal_reference_weights(
                len(source_reference), len(transfer_reference)
            )
            for mass_column in mass_columns:
                combined_values = np.concatenate([
                    source[mass_column].to_numpy(np.float64),
                    transfer[mass_column].to_numpy(np.float64),
                ])
                weights = equal_reference_weights(len(source), len(transfer))
                lower, upper = weighted_quantile(
                    combined_values, np.array([0.05, 0.95]), weights
                )
                grid = np.linspace(lower, upper, CURVE_GRID_SIZE)
                for grid_value in grid:
                    varied = reference.copy()
                    varied[mass_column] = grid_value
                    design = prepared["transform"](varied)
                    source_prediction = source_model.predict(design)
                    transfer_prediction = transfer_model.predict(design)
                    difference = transfer_prediction - source_prediction
                    curve_rows.append({
                        "analysis": analysis,
                        "target": specification["target"],
                        "feature_space": specification["feature_space"],
                        "subset": subset,
                        "half_width": half_width,
                        "mass_term": mass_column,
                        "feature_value": grid_value,
                        "predicted_flamingo": np.average(
                            source_prediction, weights=reference_weights
                        ),
                        "predicted_tng": np.average(
                            transfer_prediction, weights=reference_weights
                        ),
                        "difference_tng_minus_flamingo": np.average(
                            difference, weights=reference_weights
                        ),
                    })

flexible_summary = pd.DataFrame(flexible_summary_rows)
conditional_curves = pd.DataFrame(curve_rows)
flexible_summary.to_csv(OUTPUT / "spline_mapping_summary.csv", index=False)
conditional_curves.to_csv(OUTPUT / "conditional_mass_curves.csv", index=False)

flexible_summary.loc[
    (flexible_summary["half_width"] == PRIMARY_HALF_WIDTH)
    & (flexible_summary["status"] == "ok"),
    [
        "analysis", "subset", "n_flamingo", "n_tng",
        "adjusted_gap_tng_minus_flamingo", "rmse_flamingo", "rmse_tng",
    ],
].round(4).reset_index(drop=True)


## 6. Figures

The first figure compares adjusted simulation gaps as the score interval changes. The second
shows the component-bootstrap intervals for the primary linear estimands. The final figures
show descriptive spline differences as each mass feature is varied over the central 90% of
its equal-simulation reference distribution.

A value of zero means that the two fitted mappings give the same target at the same recorded
covariates under the stated model. It does not mean that the simulations or their catalogues
are physically identical.


In [ ]:
DISPLAY_NAMES = {
    "m200c_dynamics": "dynamics",
    "m200c_bound_ratio": "bound ratio",
    "m200c_stellar_individual": "stellar individual",
    "m200c_stellar_ratio": "stellar ratio",
    "m200c_stellar_sum": "stellar sum",
    "bound_sum_dynamics": "dynamics",
    "bound_sum_stellar_individual": "stellar individual",
    "bound_sum_stellar_ratio": "stellar ratio",
    "bound_sum_stellar_sum": "stellar sum",
}
TARGET_NAMES = {
    "m200c": "M200c",
    "bound_subhalo_sum": "Combined bound-subhalo mass",
}


fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True)
for row, target in enumerate(["m200c", "bound_subhalo_sum"]):
    for column, subset in enumerate(SUBSETS):
        ax = axes[row, column]
        current = linear_summary.loc[
            (linear_summary["target"] == target)
            & (linear_summary["subset"] == subset)
            & (linear_summary["status"] == "ok")
        ]
        for analysis in ANALYSES:
            line = current.loc[current["analysis"] == analysis].sort_values(
                "half_width"
            )
            if line.empty:
                continue
            ax.plot(
                line["half_width"],
                line["adjusted_gap_tng_minus_flamingo"],
                marker="o",
                label=DISPLAY_NAMES[analysis],
            )
        ax.axhline(0.0, color="black", linestyle="--", linewidth=1)
        ax.grid(alpha=0.25)
        if row == 0:
            ax.set_title(subset.replace("_", " ").title())
        if column == 0:
            ax.set_ylabel(f"{TARGET_NAMES[target]}\nAdjusted gap [dex]")
        if row == 1:
            ax.set_xlabel("Overlap half-width about 0.5")
axes[0, -1].legend(loc="best", fontsize=8)
fig.suptitle("Linear conditional mapping: TNG minus FLAMINGO", y=1.01)
fig.tight_layout()
fig.savefig(OUTPUT / "linear_adjusted_gap_sensitivity.png", dpi=220, bbox_inches="tight")
fig.savefig(OUTPUT / "linear_adjusted_gap_sensitivity.pdf", bbox_inches="tight")
plt.show()


for target in ["m200c", "bound_subhalo_sum"]:
    current = bootstrap_summary.loc[
        (bootstrap_summary["target"] == target)
        & (
            (bootstrap_summary["estimand"] == "adjusted_gap")
            | bootstrap_summary["is_mass_term"]
        )
    ].copy()
    current["label"] = current.apply(
        lambda row: (
            f"{DISPLAY_NAMES[row['analysis']]}: adjusted gap"
            if row["estimand"] == "adjusted_gap"
            else f"{DISPLAY_NAMES[row['analysis']]}: {row['term']}"
        ),
        axis=1,
    )
    subsets = list(SUBSETS)
    fig, axes = plt.subplots(1, 3, figsize=(15, max(5, 0.35 * len(current))))
    for ax, subset in zip(axes, subsets):
        selected = current.loc[current["subset"] == subset].reset_index(drop=True)
        positions = np.arange(len(selected))
        ax.hlines(
            positions,
            selected["lower_95"],
            selected["upper_95"],
            color="tab:blue",
            linewidth=1.5,
        )
        ax.plot(
            selected["estimate"],
            positions,
            "o",
            color="tab:blue",
        )
        ax.axvline(0.0, color="black", linestyle="--", linewidth=1)
        ax.set_yticks(positions, selected["label"], fontsize=8)
        ax.set_title(subset.replace("_", " ").title())
        ax.set_xlabel("Difference [target dex; slopes per reference SD]")
        ax.grid(axis="x", alpha=0.25)
    fig.suptitle(
        f"Component-bootstrap conditional differences: {TARGET_NAMES[target]}",
        y=1.01,
    )
    fig.tight_layout()
    stem = f"linear_component_intervals_{target}"
    fig.savefig(OUTPUT / f"{stem}.png", dpi=220, bbox_inches="tight")
    fig.savefig(OUTPUT / f"{stem}.pdf", bbox_inches="tight")
    plt.show()


for target in ["m200c", "bound_subhalo_sum"]:
    current = conditional_curves.loc[
        (conditional_curves["target"] == target)
        & (conditional_curves["subset"] == "pooled")
    ]
    panels = list(current.groupby(["analysis", "mass_term"], sort=False))
    if not panels:
        continue
    n_columns = 2
    n_rows = int(np.ceil(len(panels) / n_columns))
    fig, axes = plt.subplots(n_rows, n_columns, figsize=(12, 4 * n_rows))
    axes = np.atleast_1d(axes).ravel()
    for ax, ((analysis, mass_term), frame) in zip(axes, panels):
        ax.plot(
            frame["feature_value"],
            frame["difference_tng_minus_flamingo"],
            color="tab:purple",
            linewidth=2,
        )
        ax.axhline(0.0, color="black", linestyle="--", linewidth=1)
        ax.set_title(f"{DISPLAY_NAMES[analysis]}: {mass_term}")
        ax.set_xlabel(mass_term)
        ax.set_ylabel("Predicted target difference [dex]")
        ax.grid(alpha=0.25)
    for ax in axes[len(panels):]:
        ax.axis("off")
    fig.suptitle(
        f"Spline mapping inside 0.4--0.6: {TARGET_NAMES[target]}", y=1.01
    )
    fig.tight_layout()
    stem = f"conditional_mass_curves_{target}"
    fig.savefig(OUTPUT / f"{stem}.png", dpi=220, bbox_inches="tight")
    fig.savefig(OUTPUT / f"{stem}.pdf", bbox_inches="tight")
    plt.show()


## 7. Interpretation rules

The decision should use the linear and spline results together.

1. A non-zero adjusted gap inside `0.4--0.6` means that the fitted simulations imply
 different targets at the same reference covariates. This is evidence against simple
 marginal support mismatch as a complete explanation under the fitted mapping.
2. A mass-slope difference or a structured spline-difference curve identifies the mass
 representation associated with the conditional discrepancy. It does not establish which
 simulation is physically correct.
3. Agreement in one linear coefficient is not enough to establish transportability. Inspect
 the adjusted gap, the other controls, the flexible sensitivity and both target definitions.
4. The same-host results are better supported numerically. Different-host results must remain
 exploratory when the TNG count is small, regardless of interval width.
5. The bootstrap intervals are conditional on the selected catalogues, fixed domain scores,
 fixed feature definitions and chosen regression form. No multiplicity correction is
 applied across representations, targets or host subsets.

If a conditional discrepancy persists for absolute stellar masses but not for ratios, the
ratio representations are the stronger transport candidates. If the discrepancy persists
for all representations, the next step should examine catalogue construction and additional
joint covariates rather than expand the NPE grid. Only if the mappings become compatible
inside overlap would a targeted overlap-weighted or overlap-trained NPE be motivated.


## 8. Output manifest

The manifest fixes the input domain-score generation, population definitions, regression
settings, software versions and exported files. It records that this notebook performs no
posterior training or updating.


In [ ]:
output_manifest = {
    "experiment_id": f"conditional_mapping_{timestamp}",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "project": str(PROJECT),
    "git_revision": git_revision,
    "output": str(OUTPUT),
    "purpose": (
        "Simulation-conditional target mapping inside fixed domain-score support; "
        "no NPE training"
    ),
    "notebook": str(
        PROJECT
        / "notebooks"
        / "diagnostics"
        / "conditional_mapping_common_support.ipynb"
    ),
    "domain_result_id": DOMAIN_RESULT_ID,
    "domain_manifest_sha256": sha256(DOMAIN_MANIFEST),
    "versions": versions,
    "settings": {
        "random_seed": RANDOM_SEED,
        "bootstrap_repeats": BOOTSTRAP_REPEATS,
        "overlap_half_widths": OVERLAP_HALF_WIDTHS,
        "primary_half_width": PRIMARY_HALF_WIDTH,
        "minimum_rows_per_simulation": MIN_ROWS_PER_SIMULATION,
        "linear_reference": "equal simulation weight",
        "bootstrap_unit": "connected host component within simulation",
        "bootstrap_fixed_quantities": [
            "selected population", "domain scores", "reference centre",
            "reference scale", "linear specification",
        ],
        "spline_n_knots": SPLINE_N_KNOTS,
        "spline_degree": SPLINE_DEGREE,
        "spline_ridge_alpha": SPLINE_RIDGE_ALPHA,
        "curve_grid_size": CURVE_GRID_SIZE,
        "curve_reference_per_simulation": CURVE_REFERENCE_PER_SIMULATION,
        "validity_policy": "global_complete_case",
    },
    "analyses": ANALYSES,
    "mass_columns": MASS_COLUMNS,
    "source_population": "fixed 21,409-pair held-out FLAMINGO population",
    "transfer_population": "all 1,530 corrected TNG pairs",
    "reporting_notes": [
        "Adjusted gaps compare fitted targets at an equal-simulation reference centre.",
        "Mass slopes are in target dex per equal-reference feature standard deviation.",
        "Spline curves are descriptive and do not have bootstrap bands.",
        "Different-host results are exploratory when retained counts are small.",
        "Intervals exclude retraining, classifier uncertainty and cosmic variance.",
        "No NPE was fitted, updated or selected in this notebook.",
    ],
    "outputs": sorted(
        str(path.relative_to(OUTPUT))
        for path in OUTPUT.rglob("*")
        if path.is_file()
    ),
}
(OUTPUT / "conditional_mapping_manifest.json").write_text(
    json.dumps(output_manifest, indent=2), encoding="utf-8"
)

print(f"Completed conditional-mapping export: {OUTPUT}")
print("No NPE was trained or updated.")
